# 04 · Modelo final: perfilamiento, entrenamiento y verificación

A partir de la configuración ganadora del notebook 03 (K-Means, k=5, sobre las 11 features clínicas + síntomas gastrointestinales), este notebook reproduce el entrenamiento final, calcula las afinidades por fenotipo, guarda `data/model.pkl` y verifica que la app lo carga e infiere correctamente. Es la versión notebook de `scripts/train_model.py` (idéntica lógica, para trazabilidad).

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path('..').resolve()))
from app.domain.ml_features import FEATURE_NAMES

pd.set_option('display.width', 160)
print(FEATURE_NAMES)

['imc', 'perimetro_abdominal', 'pa_sistolica', 'pa_diastolica', 'colesterol_total', 'colesterol_hdl', 'hba1c', 'usa_hipoglicemiante', 'usa_hipolipemiante', 'n_sintomas_gi', 'usa_antiacido_ibp']


In [2]:
df = pd.read_csv('../data/_processed/02_features.csv')
df = df.rename(columns={'perimetro': 'perimetro_abdominal'})
X = df[FEATURE_NAMES].copy()
medians = X.median(numeric_only=True)
X = X.fillna(medians)

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

N_CLUSTERS = 5
RANDOM_STATE = 42
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10).fit(Xs)
sil = silhouette_score(Xs, km.labels_, sample_size=6000, random_state=RANDOM_STATE)
print(f'silueta: {sil:.4f}')
pd.Series(km.labels_).value_counts().sort_index()

silueta: 0.1857


0    4642
1    2241
2    2257
3    1636
4    5399
Name: count, dtype: int64

## Etiquetado clínico de cada cluster

En vez de fijar el índice numérico de KMeans (arbitrario y dependiente de la semilla), se etiqueta cada cluster por su rasgo dominante: el que más usa antiácidos/IBP y más síntomas GI reporta es "Digestivo"; entre los restantes, el de mayor uso de hipoglicemiantes es "Glicemia"; el de mayor uso de hipolipemiantes es "Dislipidemia"; el de mayor IMC es "Obesidad"; el que queda es "Bajo riesgo".

In [3]:
def label_clusters(df, labels):
    profile = df.assign(cluster=labels).groupby('cluster')[FEATURE_NAMES].mean()
    remaining = list(profile.index)
    mapping = {}
    digestivo = profile.loc[remaining, 'n_sintomas_gi'].idxmax()
    mapping[digestivo] = 'Digestivo'; remaining.remove(digestivo)
    glicemia = profile.loc[remaining, 'usa_hipoglicemiante'].idxmax()
    mapping[glicemia] = 'Glicemia'; remaining.remove(glicemia)
    dislipidemia = profile.loc[remaining, 'usa_hipolipemiante'].idxmax()
    mapping[dislipidemia] = 'Dislipidemia'; remaining.remove(dislipidemia)
    obesidad = profile.loc[remaining, 'imc'].idxmax()
    mapping[obesidad] = 'Obesidad'; remaining.remove(obesidad)
    assert len(remaining) == 1
    mapping[remaining[0]] = 'Bajo riesgo'
    return mapping

cluster_to_phenotype = label_clusters(df, km.labels_)
cluster_to_phenotype

{np.int32(3): 'Digestivo',
 np.int32(2): 'Glicemia',
 np.int32(1): 'Dislipidemia',
 np.int32(0): 'Obesidad',
 4: 'Bajo riesgo'}

In [4]:
phenotype_labels = pd.Series(km.labels_).map(cluster_to_phenotype)
print(phenotype_labels.value_counts())
print()
df.assign(fen=phenotype_labels.values).groupby('fen')[FEATURE_NAMES].mean().round(2).T

Bajo riesgo     5399
Obesidad        4642
Glicemia        2257
Dislipidemia    2241
Digestivo       1636
Name: count, dtype: int64



fen,Bajo riesgo,Digestivo,Dislipidemia,Glicemia,Obesidad
imc,24.67,26.16,27.64,29.52,33.33
perimetro_abdominal,81.11,86.12,90.64,96.47,101.12
pa_sistolica,115.60,117.56,118.86,120.02,120.77
pa_diastolica,75.19,76.38,77.57,78.19,78.53
colesterol_total,198.70,196.38,201.19,176.39,189.49
colesterol_hdl,54.46,51.34,49.19,46.84,43.64
hba1c,5.35,5.52,5.60,6.37,5.51
usa_hipoglicemiante,0.00,0.00,0.00,0.99,0.01
usa_hipolipemiante,0.00,0.00,1.00,0.79,0.02
n_sintomas_gi,0.96,2.09,0.05,0.02,0.29


## Afinidad por fenotipo

Para cada fenotipo, se guarda la distribución de distancias (RMS, escalada) de todo el histórico al centroide de ese cluster. En inferencia, la afinidad de un paciente nuevo es 1 menos su percentil de distancia en esa distribución — cuánto del histórico está *más lejos* del centroide que él. El fenotipo final es el de mayor afinidad.

In [5]:
dist_train = {}
for cluster_idx, phenotype in cluster_to_phenotype.items():
    dist_train[phenotype] = np.linalg.norm(Xs - km.cluster_centers_[cluster_idx], axis=1)

for phenotype, dist in dist_train.items():
    print(f'{phenotype:14s} distancia media={dist.mean():.3f}  p10={np.percentile(dist,10):.3f}  p90={np.percentile(dist,90):.3f}')

Digestivo      distancia media=4.512  p10=3.558  p90=5.856
Glicemia       distancia media=4.417  p10=3.076  p90=5.825
Dislipidemia   distancia media=3.600  p10=2.415  p90=5.048
Obesidad       distancia media=3.379  p10=1.638  p90=5.053
Bajo riesgo    distancia media=3.361  p10=1.726  p90=5.080


## Guardar el modelo

In [6]:
bundle = {
    'version': 3,
    'feature_names': FEATURE_NAMES,
    'medians': medians.to_dict(),
    'scaler': scaler,
    'kmeans': km,
    'cluster_to_phenotype': cluster_to_phenotype,
    'dist_train': dist_train,
    'silhouette': sil,
}
MODEL_PATH = Path('../data/model.pkl')
with open(MODEL_PATH, 'wb') as fh:
    pickle.dump(bundle, fh)
print('guardado en', MODEL_PATH.resolve())

guardado en C:\Hackaton2026\FenotipoMedico\data\model.pkl


## Verificación de integración con la app

`app.domain.classifier.get_classifier()` detecta `data/model.pkl` y debe devolver `TrainedClassifier` (no el `RuleBasedClassifier` de respaldo). Se prueban tres perfiles sintéticos representativos.

In [7]:
import importlib
from app.domain import classifier as classifier_module
importlib.reload(classifier_module)
clf = classifier_module.get_classifier()
print('clasificador activo:', type(clf).__name__)
assert type(clf).__name__ == 'TrainedClassifier'

clasificador activo: TrainedClassifier


In [8]:
casos = {
    'Obesidad esperada': {
        'bmi': 33.0, 'waist_cm': 102, 'sexo': 'Femenino',
        'bp_systolic': 122, 'bp_diastolic': 80,
        'colesterol_total': 185, 'hdl': 45, 'hba1c': 5.4,
        'sintomas_digestivos': [], 'medicamentos': [],
    },
    'Glicemia esperada': {
        'hba1c': 7.1, 'medicamentos': ['Hipoglicemiantes'],
        'bmi': 27, 'waist_cm': 88, 'sexo': 'Masculino',
    },
    'Digestivo esperado': {
        'bmi': 23.0, 'waist_cm': 70, 'sexo': 'Femenino',
        'sintomas_digestivos': ['Reflujo GE', 'Distensión abdominal', 'Gases'],
        'medicamentos': ['Inhibidores de la bomba de protones (Esomeprazol, lanzoprazol, pantoprazol, omeprazol)'],
    },
    'Bajo riesgo esperado': {
        'bmi': 22.0, 'waist_cm': 68, 'sexo': 'Femenino',
        'bp_systolic': 110, 'bp_diastolic': 70,
        'colesterol_total': 170, 'hdl': 60, 'hba1c': 5.0,
        'sintomas_digestivos': [], 'medicamentos': [],
    },
}
for nombre, features in casos.items():
    r = clf.predict(features)
    print(f'{nombre:22s} -> {r.phenotype:14s} scores={ {k: round(v,3) for k, v in r.scores.items()} }')

Obesidad esperada      -> Obesidad       scores={'Digestivo': 0.646, 'Glicemia': 0.76, 'Dislipidemia': 0.79, 'Obesidad': 1.0, 'Bajo riesgo': 0.66}
Glicemia esperada      -> Glicemia       scores={'Digestivo': 0.033, 'Glicemia': 0.91, 'Dislipidemia': 0.054, 'Obesidad': 0.081, 'Bajo riesgo': 0.08}
Digestivo esperado     -> Digestivo      scores={'Digestivo': 0.97, 'Glicemia': 0.009, 'Dislipidemia': 0.019, 'Obesidad': 0.019, 'Bajo riesgo': 0.107}
Bajo riesgo esperado   -> Bajo riesgo    scores={'Digestivo': 0.516, 'Glicemia': 0.204, 'Dislipidemia': 0.372, 'Obesidad': 0.299, 'Bajo riesgo': 0.836}


## Conclusiones y limitaciones

- **5 fenotipos con mejor soporte estadístico que el modelo anterior**: silueta 0.186 (k=5, clínicas+GI) frente a 0.144-0.164 (k=2-3, 19 features) del modelo anterior, y clínicamente más específicos: separan el antiguo "Cardiometabólico" en Obesidad / Dislipidemia / Glicemia, cada uno con una vía de intervención distinta.
- **Sigue sin haber un "Mixto" natural** — confirmado también con esta búsqueda más amplia de configuraciones (notebook 03). Un paciente puede tener afinidad alta a más de un fenotipo (se muestra en `scores`), pero no hay un cluster propio para esa combinación.
- **No hay variable de sexo en el histórico** para ajustar el perímetro abdominal por sexo dentro del clustering no supervisado (sí se usa para el mensaje explicativo en el clasificador de reglas de respaldo).
- **No hay *ground truth* clínico** contra el cual medir accuracy, al ser un modelo no supervisado — se recomienda validación progresiva del equipo médico vía la reclasificación manual ya disponible en la consola médica.
- **Estilo de vida se usa solo para explicar, no para clasificar**: se probó incluirlo en el clustering (notebook 03) y empeoraba la silueta y la interpretabilidad; sigue disponible para el paciente vía `rationale`.